# 🧪 F-18 Model Testing Notebook

**Amaç:** Eğitilmiş modelleri test et ve karşılaştır.

**Kullanım:**
1. Checkpoint dosyalarını Kaggle'a yükle (Dataset olarak)
2. `MODEL_PATHS` ve `SELECTED_EPOCHS` ayarla
3. Hücreleri sırayla çalıştır

**Kaggle için optimize edilmiştir!**

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🎛️ KONFİGÜRASYON - BURADAN AYARLA!
# ══════════════════════════════════════════════════════════════

# ═══ ANA DATASET KLASÖRÜ ═══
DATASET_ROOT = '/kaggle/input/modelsss/modeltesting'

# ═══ CHECKPOINT KAYNAKLARI ═══
# Kaggle'ın kendi output'undan (iptal olan eğitim)
KAGGLE_INPUT_DIR = f'{DATASET_ROOT}/kaggle-input'

# Colab/Drive'dan yüklenen checkpoint'ler
COLAB_INPUT_DIR = f'{DATASET_ROOT}/colab-input'

# Gerçek görüntüler için klasör
REAL_GRAPHS_DIR = f'{DATASET_ROOT}/real-graph'

# Hangisini kullan? 'kaggle', 'colab' veya 'both'
USE_SOURCE = 'both'

# Test edilecek epoch'lar (None = otomatik seç)
SELECTED_EPOCHS = None  # Örnek: [78, 50, 25, 10] veya None

# Best model dosya adları (her iki kaynak için)
BEST_MODEL_NAMES = ['best_colored.pt', 'best_model.pt']

# Her test türünde kaç görüntü üretilsin?
IMAGES_PER_TEST = 5

# Görüntü boyutu
IMG_SIZE = 512

# Çıktı klasörü
OUTPUT_DIR = '/kaggle/working/test_results'

print("✅ Konfigürasyon hazır!")
print(f"   📁 Dataset Root: {DATASET_ROOT}")
print(f"   📁 Kaggle Input: {KAGGLE_INPUT_DIR}")
print(f"   📁 Colab Input: {COLAB_INPUT_DIR}")
print(f"   📁 Real Graphs: {REAL_GRAPHS_DIR}")
print(f"   🎯 Kaynak: {USE_SOURCE}")
print(f"   📊 Test başına görüntü: {IMAGES_PER_TEST}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 📦 IMPORTS
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
from dataclasses import dataclass
import random
import math
import io
import os
import glob
import re
from tqdm.auto import tqdm

# Kaggle/Colab için inline görüntüleme
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️ Device: {device}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🏗️ U-NET MODEL
# ══════════════════════════════════════════════════════════════

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)
        
        for f in features:
            self.downs.append(DoubleConv(in_channels, f))
            in_channels = f
        
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.ups.append(DoubleConv(f * 2, f))
        
        self.final = nn.Conv2d(features[0], out_channels, 1)
    
    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
        
        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]
        
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skip_connections[i // 2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i + 1](x)
        
        return torch.sigmoid(self.final(x))

print("✅ U-Net model tanımlandı")
print(f"   Parametreler: {sum(p.numel() for p in UNet().parameters()):,}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 📊 SENTETİK VERİ ÜRETİCİ (EĞİTİM İLE AYNI!)
# ══════════════════════════════════════════════════════════════

@dataclass
class ChartConfig:
    """Grafik konfigürasyonu - EĞİTİM İLE BİREBİR AYNI"""
    x_min: float = 0.30
    x_max: float = 1.00
    y_min: float = 0.04
    y_max: float = 0.15
    n_curves: int = 8
    curve_type: str = 'peaked'
    curve_lw: float = 0.6  # Eğitim: 0.3-0.6 uniform
    add_grid: bool = True
    add_arrows: bool = True
    add_envelope_optimum: bool = True
    add_envelope_endurance: bool = False  # Eğitim: %35
    add_vmax_line: bool = False  # Eğitim: %25
    add_text_boxes: bool = True
    add_fuel_labels: bool = True
    add_drag_labels: bool = True  # Eğitim: %55


def generate_curve_shape(x, curve_type, curve_index, total_curves):
    """Eğri şekli üret - EĞİTİM İLE BİREBİR AYNI"""
    alt = curve_index / max(total_curves - 1, 1)
    x_norm = (x - x.min()) / (x.max() - x.min() + 1e-8)
    
    if curve_type == 'peaked':
        peak_pos = 0.30 + random.uniform(-0.05, 0.05)
        start_y = 0.12 + random.uniform(-0.02, 0.02)
        peak_y = 0.45 + alt * 0.40 + random.uniform(-0.03, 0.03)
        end_y = 0.20 + alt * 0.25 + random.uniform(-0.02, 0.02)
        y = np.zeros_like(x_norm)
        for i, t in enumerate(x_norm):
            if t <= peak_pos:
                progress = t / peak_pos
                y[i] = start_y + (peak_y - start_y) * (1 - (1 - progress) ** 2)
            else:
                progress = (t - peak_pos) / (1 - peak_pos)
                y[i] = peak_y - (peak_y - end_y) * (progress ** 0.7)
                
    elif curve_type == 'peaked_oval':
        peak_pos = 0.45 + random.uniform(-0.07, 0.07)
        start_y = 0.12 + random.uniform(-0.02, 0.02)
        peak_y = 0.45 + alt * 0.38 + random.uniform(-0.03, 0.03)
        end_y = 0.20 + alt * 0.25 + random.uniform(-0.02, 0.02)
        y = np.zeros_like(x_norm)
        for i, t in enumerate(x_norm):
            if t <= peak_pos:
                progress = t / peak_pos
                y[i] = start_y + (peak_y - start_y) * (math.sin(progress * math.pi / 2) ** 1.2)
            else:
                progress = (t - peak_pos) / (1 - peak_pos)
                y[i] = end_y + (peak_y - end_y) * (math.cos(progress * math.pi / 2) ** 1.2)
        
    elif curve_type == 'wavy':
        freq = random.choice([1.0, 1.5, 2.0])
        phase = random.uniform(0, 1)
        wave = 0.5 + 0.25 * np.sin(2 * np.pi * (x_norm * freq + phase))
        wave += 0.12 * np.sin(4 * np.pi * (x_norm * freq + phase))
        hump_center = random.uniform(0.55, 0.70)
        hump = 0.12 * np.exp(-((x_norm - hump_center) / 0.22) ** 2)
        y = wave + hump
        y = np.clip(y, 0.05, 0.95)
        
    elif curve_type == 'rising':
        start_y = 0.08 + alt * 0.05 + random.uniform(-0.02, 0.02)
        end_y = 0.55 + alt * 0.30 + random.uniform(-0.03, 0.03)
        curvature = random.uniform(0.7, 1.3)
        y = start_y + (end_y - start_y) * (x_norm ** curvature)
        
    elif curve_type == 'falling':
        start_y = 0.65 + alt * 0.25 + random.uniform(-0.03, 0.03)
        end_y = 0.12 + alt * 0.10 + random.uniform(-0.02, 0.02)
        curvature = random.uniform(0.5, 1.0)
        y = start_y - (start_y - end_y) * (x_norm ** curvature)
    
    elif curve_type == 'mixed':
        # EĞİTİMDE VAR! - Rastgele tip seç
        return generate_curve_shape(x, random.choice(['peaked', 'peaked_oval', 'rising', 'falling', 'wavy']),
                                   curve_index, total_curves)
    else:
        return generate_curve_shape(x, random.choice(['peaked', 'peaked_oval', 'rising', 'falling', 'wavy']),
                                   curve_index, total_curves)
    return y


def fig_to_array(fig, dpi=150, tight=True):
    """Figure'ı numpy array'e çevir"""
    buf = io.BytesIO()
    if tight:
        fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', pad_inches=0.02,
                    facecolor='white', edgecolor='none')
    else:
        fig.savefig(buf, format='png', dpi=dpi, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf).convert('RGB')
    return np.array(img)

print("✅ Sentetik veri üretici yüklendi (EĞİTİM İLE AYNI!)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🎨 GRAFİK ÇİZİM FONKSİYONU (EĞİTİM İLE AYNI!)
# ══════════════════════════════════════════════════════════════

def draw_chart_matplotlib(config, W=512, H=512):
    """Grafik çiz ve renkli mask üret - EĞİTİM İLE BİREBİR AYNI"""
    fig_w, fig_h = W / 100, H / 100
    
    x = np.linspace(config.x_min + 0.02, config.x_max - 0.02, 400)
    curves_data = []
    for i in range(config.n_curves):
        y_norm = generate_curve_shape(x, config.curve_type, i, config.n_curves)
        y = config.y_min + y_norm * (config.y_max - config.y_min)
        y = np.clip(y, config.y_min + 0.001, config.y_max - 0.001)
        curves_data.append((x.copy(), y))
    
    # ========== FULL IMAGE ==========
    fig1, ax1 = plt.subplots(figsize=(fig_w, fig_h))
    ax1.set_xlim(config.x_min, config.x_max)
    ax1.set_ylim(config.y_min, config.y_max)
    
    # Grid - Minor grid dahil
    if config.add_grid:
        x_range = config.x_max - config.x_min
        y_range = config.y_max - config.y_min
        x_major = 0.1 if x_range > 0.5 else 0.05
        y_major = 0.01 if y_range < 0.08 else 0.02
        ax1.set_xticks(np.arange(config.x_min, config.x_max + 0.001, x_major))
        ax1.set_xticks(np.arange(config.x_min, config.x_max + 0.001, x_major/2), minor=True)
        ax1.set_yticks(np.arange(config.y_min, config.y_max + 0.001, y_major))
        ax1.set_yticks(np.arange(config.y_min, config.y_max + 0.001, y_major/2), minor=True)
        ax1.grid(True, which='major', linewidth=0.8, alpha=0.5, color='black')
        ax1.grid(True, which='minor', linewidth=0.4, alpha=0.3, color='black')
    
    ax1.axhline(y=config.y_min, color='black', linewidth=2.0, zorder=10)
    ax1.axvline(x=config.x_min, color='black', linewidth=2.0, zorder=10)
    ax1.tick_params(axis='both', which='major', length=6, width=1.5, direction='in')
    ax1.tick_params(axis='both', which='minor', length=3, width=1.0, direction='in')
    
    for spine in ax1.spines.values():
        spine.set_linewidth(1.5)
    
    ax1.set_xlabel('MACH NUMBER', fontsize=10, fontweight='bold')
    ax1.set_ylabel('SPECIFIC RANGE — NAUTICAL MILES PER POUND OF FUEL', fontsize=8)
    
    for cx, cy in curves_data:
        ax1.plot(cx, cy, 'k-', linewidth=config.curve_lw)
    
    # OPTIMUM CRUISE envelope
    if config.add_envelope_optimum:
        if config.curve_type in ['peaked', 'peaked_oval']:
            envelope_pts = [(cx[np.argmax(cy)], cy.max()) for cx, cy in curves_data]
        else:
            envelope_pts = [(cx[int(len(cx)*0.5)], cy[int(len(cy)*0.5)]) for cx, cy in curves_data]
        envelope_pts.sort(key=lambda p: p[1])
        ex, ey = zip(*envelope_pts)
        ax1.plot(ex, ey, 'k-', linewidth=1.2)
        ax1.text(ex[0] - 0.03, ey[-1] + (config.y_max - config.y_min) * 0.02,
                'OPTIMUM\nCRUISE', fontsize=8, ha='right', va='bottom')
    
    # MAXIMUM ENDURANCE envelope (EĞİTİMDE VAR!)
    if config.add_envelope_endurance:
        envelope_pts = [(cx[int(len(cx)*0.2)], cy[int(len(cy)*0.2)]) for cx, cy in curves_data]
        envelope_pts.sort(key=lambda p: p[1])
        ex, ey = zip(*envelope_pts)
        ax1.plot(ex, ey, 'k-', linewidth=1.2)
        ax1.text(ex[-1] - 0.02, ey[0] - (config.y_max - config.y_min) * 0.02,
                'MAXIMUM\nENDURANCE', fontsize=8, ha='right', va='top')
    
    # ARROWS - Geliştirilmiş (EĞİTİM İLE AYNI)
    if config.add_arrows:
        fuel_flows = ['3000', '3500', '4000', '4500', '5000', '5500', '6000', '6500', '7000', '7500', '8000', '8500']
        for idx, (cx, cy) in enumerate(reversed(curves_data)):
            if idx >= len(fuel_flows):
                break
            
            if random.random() < 0.5:
                arrow_idx = -1
                x_head = cx[arrow_idx]
                y_head = cy[arrow_idx]
                dx = random.uniform(0.04, 0.08)
                dy = random.uniform(-0.005, 0.005)
                x_tail = x_head + dx
                y_tail = y_head + dy
            else:
                mid_start = len(cx) // 4
                mid_end = 3 * len(cx) // 4
                arrow_idx = random.randint(mid_start, mid_end)
                x_head = cx[arrow_idx]
                y_head = cy[arrow_idx]
                angle = random.uniform(20, 70)
                dist = random.uniform(0.05, 0.10)
                if random.random() < 0.5:
                    dx = dist * math.cos(math.radians(angle))
                    dy = dist * math.sin(math.radians(angle))
                else:
                    dx = dist * math.cos(math.radians(-angle))
                    dy = dist * math.sin(math.radians(-angle))
                x_tail = x_head + dx
                y_tail = y_head + dy
            
            ax1.plot([x_tail, x_head], [y_tail, y_head], color="black", linewidth=0.6)
            
            if random.random() < 0.4:
                arrow_style = random.choice(["-|>", "->"])
                fill_style = "none"
            else:
                arrow_style = random.choice(["-|>", "-|>", "->"])
                fill_style = "black"
            
            ax1.annotate(
                "",
                xy=(x_head, y_head),
                xytext=(x_tail, y_tail),
                arrowprops=dict(
                    arrowstyle=arrow_style,
                    lw=random.uniform(0.7, 1.1),
                    color="black",
                    fc=fill_style,
                    shrinkA=0, shrinkB=0,
                    mutation_scale=random.uniform(12, 18),
                ),
            )
            
            # Dashed leader line (kesikli çizgi) - EĞİTİMDE VAR
            if random.random() < 0.85:
                dash_len = random.uniform(0.06, 0.14)
                dash_angle = math.radians(random.choice([35, 45, 55, 65, 75]))
                dash_dx = dash_len * math.cos(dash_angle)
                dash_dy = dash_len * math.sin(dash_angle)
                base_x = config.x_min + (config.x_max - config.x_min) * random.uniform(0.25, 0.75)
                base_y = config.y_min + (config.y_max - config.y_min) * random.uniform(0.25, 0.75)
                ax1.plot([base_x, base_x + dash_dx], [base_y, base_y + dash_dy],
                        color="black", linewidth=0.6, linestyle=(0, (14, 8)))
            
            label_x = x_tail + random.uniform(0.05, 0.09)
            ax1.text(label_x, y_tail + random.uniform(-0.002, 0.002),
                    fuel_flows[idx], fontsize=8, va='center', ha='left')
    
    # Additional standalone dashed lines - EĞİTİMDE VAR
    if random.random() < 0.7:
        n_extra_dashes = random.randint(2, 6)
        for _ in range(n_extra_dashes):
            dcx = config.x_min + (config.x_max - config.x_min) * random.uniform(0.2, 0.8)
            dcy = config.y_min + (config.y_max - config.y_min) * random.uniform(0.2, 0.8)
            dash_len = random.uniform(0.04, 0.10)
            dash_angle = math.radians(random.choice([35, 45, 55, 65, 75]))
            dash_dx = dash_len * math.cos(dash_angle)
            dash_dy = dash_len * math.sin(dash_angle)
            ax1.plot([dcx, dcx + dash_dx], [dcy, dcy + dash_dy],
                    color="black", linewidth=0.5, linestyle=(0, (12, 7)))
    
    # Text boxes
    if config.add_text_boxes:
        ax1.text(config.x_max - 0.05, config.y_max - 0.005,
                'TOTAL FUEL FLOW—\nPOUNDS PER HOUR',
                fontsize=8, ha='right', va='top',
                bbox=dict(boxstyle='square,pad=0.3', facecolor='white', edgecolor='black'))
        
        # Legend box - EĞİTİMDE VAR
        legend_x = config.x_min + (config.x_max - config.x_min) * 0.15
        legend_y = config.y_max - (config.y_max - config.y_min) * 0.1
        ax1.text(legend_x, legend_y,
                '◄─ CRUISE    DASH ─►\n      AOA          AOA\n(USED FOR INTERFERENCE\n DRAG DETERMINATION)',
                fontsize=7, ha='left', va='top',
                bbox=dict(boxstyle='square,pad=0.3', facecolor='white', edgecolor='black'))
    
    # Drag index labels - EĞİTİMDE VAR
    if config.add_drag_labels:
        labels = ['0.00', '25.00', '50.00', '75.00', '100.00', '125.00', '150.00']
        base_x = config.x_min + (config.x_max - config.x_min) * 0.65
        base_y = config.y_min + (config.y_max - config.y_min) * 0.15
        for i, lbl in enumerate(labels[:random.randint(4, 7)]):
            ax1.text(base_x + random.uniform(-0.02, 0.02),
                    base_y + i * (config.y_max - config.y_min) * 0.05,
                    lbl, fontsize=7, alpha=0.9)
    
    # Vmax line - EĞİTİMDE VAR
    if config.add_vmax_line:
        vmax_pts = [(cx[int(len(cx)*0.85)], cy[int(len(cy)*0.85)]) for cx, cy in curves_data]
        vmax_pts.sort(key=lambda p: p[1])
        vx, vy = zip(*vmax_pts)
        ax1.plot(vx, vy, 'k--', linewidth=0.8)
        ax1.text(vx[-1], vy[-1] + 0.003, r'$V_{max}$(MIL)', fontsize=7)
    
    full_img = fig_to_array(fig1, dpi=150, tight=True)
    full_img = cv2.resize(full_img, (W, H))
    
    # ========== COLORED MASK (RGB, HSV renk sistemi ile) ==========
    fig2, ax2 = plt.subplots(figsize=(fig_w, fig_h))
    ax2.set_xlim(config.x_min, config.x_max)
    ax2.set_ylim(config.y_min, config.y_max)
    ax2.set_position([0, 0, 1, 1])
    ax2.axis('off')
    fig2.patch.set_facecolor('black')
    ax2.set_facecolor('black')
    
    n_curves = len(curves_data)
    for i, (cx, cy) in enumerate(curves_data):
        hue = int(180 * i / max(n_curves, 1))
        hsv_color = np.array([[[hue, 255, 255]]], dtype=np.uint8)
        bgr_color = cv2.cvtColor(hsv_color, cv2.COLOR_HSV2BGR)[0, 0]
        rgb_color = (int(bgr_color[2]), int(bgr_color[1]), int(bgr_color[0]))
        # Mask linewidth: input ile orantılı (+0.2) - EĞİTİM İLE AYNI
        ax2.plot(cx, cy, color=np.array(rgb_color) / 255.0, linewidth=config.curve_lw + 0.2, zorder=2)
    
    colored_img = fig_to_array(fig2, dpi=150, tight=False)
    colored_img = cv2.resize(colored_img, (W, H))
    
    return full_img, colored_img, curves_data


def add_scan_artifacts(img, strength=1.0):
    """Scan/photocopy artifacts ekle - EĞİTİM İLE AYNI"""
    pil_img = Image.fromarray(img)
    angle = random.uniform(-1.2, 1.2) * strength
    pil_img = pil_img.rotate(angle, fillcolor=(255, 255, 255), resample=Image.BICUBIC)
    pil_img = ImageEnhance.Brightness(pil_img).enhance(random.uniform(0.90, 1.10))
    pil_img = ImageEnhance.Contrast(pil_img).enhance(random.uniform(0.88, 1.12))
    arr = np.array(pil_img).astype(np.float32) / 255.0
    noise = np.random.normal(0, 0.012 * strength, arr.shape)
    arr = np.clip(arr + noise, 0, 1)
    buf = io.BytesIO()
    Image.fromarray((arr * 255).astype(np.uint8)).save(buf, format='JPEG', quality=random.randint(50, 80))
    buf.seek(0)
    return np.array(Image.open(buf).convert('RGB'))


def random_config(curve_type=None):
    """Rastgele konfigürasyon üret - EĞİTİM İLE AYNI PARAMETRELER!"""
    # EĞİTİMDEKİ RANGES (7 adet)
    x_ranges = [
        (0.30, 0.95), (0.30, 1.00), (0.40, 1.10), (0.50, 1.20),
        (0.50, 1.30), (0.50, 1.40), (0.60, 1.40)
    ]
    y_ranges = [
        (0.04, 0.15), (0.05, 0.15), (0.06, 0.17), (0.07, 0.18),
        (0.08, 0.19), (0.08, 0.20), (0.05, 0.14)
    ]
    
    x_min, x_max = random.choice(x_ranges)
    y_min, y_max = random.choice(y_ranges)
    
    # EĞİTİMDEKİ CURVE TYPES (6 tip - mixed dahil!)
    if curve_type is None:
        curve_types = ['peaked_oval'] * 28 + ['peaked'] * 26 + ['rising'] * 16 + ['falling'] * 14 + ['wavy'] * 10 + ['mixed'] * 6
        curve_type = random.choice(curve_types)
    
    if curve_type == 'wavy':
        n_curves = random.randint(3, 6)
    elif curve_type in ['falling', 'mixed']:
        n_curves = random.randint(4, 7)
    elif curve_type == 'rising':
        n_curves = random.randint(5, 8)
    else:
        n_curves = random.randint(6, 12)
    
    return ChartConfig(
        x_min=x_min, x_max=x_max,
        y_min=y_min, y_max=y_max,
        n_curves=n_curves,
        curve_type=curve_type,
        curve_lw=random.uniform(0.3, 0.6),  # EĞİTİM İLE AYNI
        add_grid=random.random() < 0.95,
        add_arrows=random.random() < 0.85,
        add_envelope_optimum=random.random() < 0.70,
        add_envelope_endurance=random.random() < 0.35,  # EĞİTİMDE VAR!
        add_vmax_line=random.random() < 0.25,  # EĞİTİMDE VAR!
        add_text_boxes=random.random() < 0.75,
        add_fuel_labels=random.random() < 0.80,
        add_drag_labels=random.random() < 0.55,  # EĞİTİMDE VAR!
    )

print("✅ Grafik çizim fonksiyonları yüklendi (EĞİTİM İLE AYNI!)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🔍 CHECKPOINT'LERİ BUL VE LİSTELE
# ══════════════════════════════════════════════════════════════

def extract_epoch(path):
    match = re.search(r'epoch_?(\d+)', os.path.basename(path))
    return int(match.group(1)) if match else 0

# Checkpoint dosyalarını bul
checkpoint_dict = {}
best_models = []  # Birden fazla best model olabilir

# Kaynaklara göre checkpoint'leri topla
sources_to_check = []
if USE_SOURCE in ['kaggle', 'both']:
    sources_to_check.append(('Kaggle', KAGGLE_INPUT_DIR))
if USE_SOURCE in ['colab', 'both']:
    sources_to_check.append(('Colab', COLAB_INPUT_DIR))

print("🔍 Checkpoint'ler aranıyor...\n")

for source_name, source_dir in sources_to_check:
    if not os.path.exists(source_dir):
        print(f"   ⚠️ {source_name}: Klasör bulunamadı ({source_dir})")
        continue
    
    # .pt dosyalarını bul
    pt_files = glob.glob(f'{source_dir}/**/*.pt', recursive=True)
    print(f"   📁 {source_name}: {len(pt_files)} checkpoint bulundu")
    
    for p in pt_files:
        basename = os.path.basename(p)
        
        # Best model kontrolü
        is_best = any(best_name in basename for best_name in BEST_MODEL_NAMES)
        if is_best:
            best_models.append({'path': p, 'source': source_name, 'name': basename})
            print(f"      🏆 Best model: {basename} ({source_name})")
            continue
        
        # Epoch numarasını çıkar
        ep = extract_epoch(p)
        if ep > 0:
            key = (ep, source_name)
            checkpoint_dict[key] = {'path': p, 'source': source_name, 'epoch': ep}

# Epoch'ları listele
all_epochs = sorted(set(k[0] for k in checkpoint_dict.keys()))

print(f"\n📊 Özet:")
print(f"   Toplam unique epoch: {len(all_epochs)}")
if all_epochs:
    print(f"   Epoch aralığı: {min(all_epochs)} - {max(all_epochs)}")
print(f"   Best model sayısı: {len(best_models)}")

# Test edilecek modelleri belirle
if SELECTED_EPOCHS is None:
    # Otomatik: son, son-5, son-10, son-20, son-40
    last_epoch = max(all_epochs) if all_epochs else 0
    auto_epochs = [last_epoch]
    for offset in [5, 10, 20, 40]:
        target = last_epoch - offset
        if target in all_epochs:
            auto_epochs.append(target)
        else:
            closest = min(all_epochs, key=lambda x: abs(x - target)) if all_epochs else 0
            if closest not in auto_epochs and closest > 0:
                auto_epochs.append(closest)
    test_epochs = sorted(set(auto_epochs), reverse=True)[:5]
else:
    test_epochs = [e for e in SELECTED_EPOCHS if e in all_epochs]

print(f"\n📋 Test edilecek epoch'lar: {test_epochs}")

# Model listesi oluştur
models_to_test = []

# Best modelleri ekle
for bm in best_models:
    prefix = '🟦' if bm['source'] == 'Kaggle' else '🟩'
    models_to_test.append({
        'name': f"🏆 {prefix} {bm['source']} BEST",
        'path': bm['path'],
        'epoch': 'best',
        'source': bm['source']
    })

# Epoch'ları ekle (her kaynaktan)
for ep in test_epochs:
    for source_name, _ in sources_to_check:
        key = (ep, source_name)
        if key in checkpoint_dict:
            info = checkpoint_dict[key]
            prefix = '🟦' if source_name == 'Kaggle' else '🟩'
            models_to_test.append({
                'name': f'{prefix} {source_name} E{ep}',
                'path': info['path'],
                'epoch': ep,
                'source': source_name
            })

print(f"\n🔬 Test edilecek modeller ({len(models_to_test)}):\n")
for m in models_to_test:
    print(f"   • {m['name']}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🔧 MODEL YÜKLEME & TAHMİN FONKSİYONLARI
# ══════════════════════════════════════════════════════════════

def load_model(path, device):
    """Checkpoint'ten model yükle"""
    ckpt = torch.load(path, map_location=device)
    model = UNet(in_channels=3, out_channels=3)
    
    if isinstance(ckpt, dict):
        if 'model_state_dict' in ckpt:
            model.load_state_dict(ckpt['model_state_dict'])
        elif 'state_dict' in ckpt:
            model.load_state_dict(ckpt['state_dict'])
        else:
            model.load_state_dict(ckpt)
    else:
        model.load_state_dict(ckpt)
    
    model.to(device)
    model.eval()
    return model


def predict(model, img_rgb, device):
    """Tahmin yap (raw + görsel)"""
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0
    
    with torch.no_grad():
        pred = model(img_tensor.unsqueeze(0).to(device))
        pred = pred.squeeze(0).cpu().permute(1, 2, 0).numpy()
        pred_raw = np.clip(pred, 0, 1)
        
        # Görsel için kontrast artır
        pred_vis = pred_raw.copy()
        pmax = pred_vis.max()
        if pmax > 1e-6:
            pred_vis = pred_vis / pmax
        
        pred_raw = (pred_raw * 255).astype(np.uint8)
        pred_vis = (pred_vis * 255).astype(np.uint8)
    return pred_raw, pred_vis


def calculate_metrics(pred, target):
    """MAE ve benzerlikleri hesapla"""
    mae = np.abs(pred.astype(float) - target.astype(float)).mean()
    
    # Binary mask comparison (eğri var mı yok mu)
    pred_gray = cv2.cvtColor(pred, cv2.COLOR_RGB2GRAY)
    target_gray = cv2.cvtColor(target, cv2.COLOR_RGB2GRAY)
    pred_bin = (pred_gray > 30).astype(np.uint8)
    target_bin = (target_gray > 30).astype(np.uint8)
    
    intersection = (pred_bin & target_bin).sum()
    union = (pred_bin | target_bin).sum()
    iou = intersection / (union + 1e-8)
    
    return {'mae': mae, 'iou': iou}

print("✅ Model yükleme fonksiyonları hazır")

---
## 🧪 TEST 1: PEAKED_OVAL (En Yaygın Tip)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🧪 TEST 1: PEAKED_OVAL - MODEL TAHMİN KARŞILAŞTIRMASI
# ══════════════════════════════════════════════════════════════

print("="*70)
print("🧪 TEST 1: PEAKED_OVAL - MODEL TAHMİN KARŞILAŞTIRMASI")
print("="*70)

curve_type = 'peaked_oval'
n_images = IMAGES_PER_TEST
n_models = len(models_to_test)

# Modelleri yükle
print("\n📦 Modeller yükleniyor...")
loaded_models = {}
for m in models_to_test:
    try:
        loaded_models[m['name']] = load_model(m['path'], device)
        print(f"   ✅ {m['name']}")
    except Exception as e:
        print(f"   ❌ {m['name']}: {e}")
        loaded_models[m['name']] = None

print(f"\n🎨 {n_images} sentetik görüntü üretiliyor ve test ediliyor...\n")

# Her görüntü için ayrı figure
for img_idx in range(n_images):
    print(f"📷 [{img_idx+1}/{n_images}] Görüntü üretiliyor...")
    
    # Sentetik veri üret
    cfg = random_config(curve_type=curve_type)
    img, target, _ = draw_chart_matplotlib(cfg, W=IMG_SIZE, H=IMG_SIZE)
    img_noisy = add_scan_artifacts(img, strength=1.0)
    
    # Figure: Input + Ground Truth + Model tahminleri
    fig, axes = plt.subplots(1, n_models + 2, figsize=(4 * (n_models + 2), 5))
    
    # INPUT
    axes[0].imshow(img_noisy)
    axes[0].set_title('📥 INPUT\n(Gürültülü Grafik)', fontsize=11, fontweight='bold', color='blue')
    axes[0].axis('off')
    
    # GROUND TRUTH (Hedef)
    axes[1].imshow(target)
    axes[1].set_title('🎯 GROUND TRUTH\n(Hedef Mask)', fontsize=11, fontweight='bold', color='green')
    axes[1].axis('off')
    
    # MODEL TAHMİNLERİ
    for col, m in enumerate(models_to_test):
        model = loaded_models.get(m['name'])
        
        if model is not None:
            try:
                pred_raw, pred_vis = predict(model, img_noisy, device)
                metrics = calculate_metrics(pred_raw, target)
                
                axes[col + 2].imshow(pred_vis)
                
                if 'BEST' in m['name']:
                    title = f"🏆 PREDICTION\n{m['source']} BEST"
                    color = 'purple'
                else:
                    title = f"📤 PREDICTION\n{m['name']}"
                    color = 'darkblue' if 'Kaggle' in m['name'] else 'darkgreen'
                
                axes[col + 2].set_title(f"{title}\nMAE:{metrics['mae']:.1f} IoU:{metrics['iou']:.2f}", 
                                        fontsize=9, fontweight='bold', color=color)
                axes[col + 2].axis('off')
            except Exception as e:
                axes[col + 2].text(0.5, 0.5, f'Hata', ha='center', va='center', fontsize=10)
                axes[col + 2].axis('off')
        else:
            axes[col + 2].text(0.5, 0.5, 'Model Yok', ha='center', va='center', fontsize=10)
            axes[col + 2].axis('off')
    
    plt.suptitle(f'🧪 TEST 1: {curve_type.upper()} #{img_idx+1}\n'
                 f'[INPUT] → [GROUND TRUTH] → [MODEL PREDICTIONS]', 
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    save_path = f'{OUTPUT_DIR}/test1_{curve_type}_{img_idx+1}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"   💾 Kaydedildi: {os.path.basename(save_path)}")

# Memory temizle
for name, model in loaded_models.items():
    if model is not None:
        del model
torch.cuda.empty_cache()

print(f"\n✅ TEST 1 TAMAMLANDI - {n_images} görüntü test edildi")

---
## 🧪 TEST 2: PEAKED (Sivri Tepe)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🧪 TEST 2: PEAKED - MODEL TAHMİN KARŞILAŞTIRMASI
# ══════════════════════════════════════════════════════════════

print("="*70)
print("🧪 TEST 2: PEAKED - MODEL TAHMİN KARŞILAŞTIRMASI")
print("="*70)

curve_type = 'peaked'
n_images = IMAGES_PER_TEST
n_models = len(models_to_test)

print("\n📦 Modeller yükleniyor...")
loaded_models = {}
for m in models_to_test:
    try:
        loaded_models[m['name']] = load_model(m['path'], device)
    except:
        loaded_models[m['name']] = None

print(f"🎨 {n_images} sentetik görüntü test ediliyor...\n")

for img_idx in range(n_images):
    cfg = random_config(curve_type=curve_type)
    img, target, _ = draw_chart_matplotlib(cfg, W=IMG_SIZE, H=IMG_SIZE)
    img_noisy = add_scan_artifacts(img, strength=1.0)
    
    fig, axes = plt.subplots(1, n_models + 2, figsize=(4 * (n_models + 2), 5))
    
    axes[0].imshow(img_noisy)
    axes[0].set_title('📥 INPUT\n(Gürültülü Grafik)', fontsize=11, fontweight='bold', color='blue')
    axes[0].axis('off')
    
    axes[1].imshow(target)
    axes[1].set_title('🎯 GROUND TRUTH\n(Hedef Mask)', fontsize=11, fontweight='bold', color='green')
    axes[1].axis('off')
    
    for col, m in enumerate(models_to_test):
        model = loaded_models.get(m['name'])
        if model is not None:
            try:
                pred_raw, pred_vis = predict(model, img_noisy, device)
                metrics = calculate_metrics(pred_raw, target)
                axes[col + 2].imshow(pred_vis)
                title = f"🏆 PRED\n{m['source']} BEST" if 'BEST' in m['name'] else f"📤 PRED\n{m['name']}"
                color = 'purple' if 'BEST' in m['name'] else ('darkblue' if 'Kaggle' in m['name'] else 'darkgreen')
                axes[col + 2].set_title(f"{title}\nMAE:{metrics['mae']:.1f}", fontsize=9, fontweight='bold', color=color)
            except:
                axes[col + 2].text(0.5, 0.5, 'Hata', ha='center', va='center')
        else:
            axes[col + 2].text(0.5, 0.5, 'Model Yok', ha='center', va='center')
        axes[col + 2].axis('off')
    
    plt.suptitle(f'🧪 TEST 2: {curve_type.upper()} #{img_idx+1}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/test2_{curve_type}_{img_idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

for model in loaded_models.values():
    if model: del model
torch.cuda.empty_cache()
print(f"✅ TEST 2 TAMAMLANDI")

---
## 🧪 TEST 3: RISING (Yükselen)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🧪 TEST 3: RISING - MODEL TAHMİN KARŞILAŞTIRMASI
# ══════════════════════════════════════════════════════════════

print("="*70)
print("🧪 TEST 3: RISING - MODEL TAHMİN KARŞILAŞTIRMASI")
print("="*70)

curve_type = 'rising'
n_images = IMAGES_PER_TEST
n_models = len(models_to_test)

print("\n📦 Modeller yükleniyor...")
loaded_models = {}
for m in models_to_test:
    try:
        loaded_models[m['name']] = load_model(m['path'], device)
    except:
        loaded_models[m['name']] = None

print(f"🎨 {n_images} sentetik görüntü test ediliyor...\n")

for img_idx in range(n_images):
    cfg = random_config(curve_type=curve_type)
    img, target, _ = draw_chart_matplotlib(cfg, W=IMG_SIZE, H=IMG_SIZE)
    img_noisy = add_scan_artifacts(img, strength=1.0)
    
    fig, axes = plt.subplots(1, n_models + 2, figsize=(4 * (n_models + 2), 5))
    
    axes[0].imshow(img_noisy)
    axes[0].set_title('📥 INPUT\n(Gürültülü Grafik)', fontsize=11, fontweight='bold', color='blue')
    axes[0].axis('off')
    
    axes[1].imshow(target)
    axes[1].set_title('🎯 GROUND TRUTH\n(Hedef Mask)', fontsize=11, fontweight='bold', color='green')
    axes[1].axis('off')
    
    for col, m in enumerate(models_to_test):
        model = loaded_models.get(m['name'])
        if model is not None:
            try:
                pred_raw, pred_vis = predict(model, img_noisy, device)
                metrics = calculate_metrics(pred_raw, target)
                axes[col + 2].imshow(pred_vis)
                title = f"🏆 PRED\n{m['source']} BEST" if 'BEST' in m['name'] else f"📤 PRED\n{m['name']}"
                color = 'purple' if 'BEST' in m['name'] else ('darkblue' if 'Kaggle' in m['name'] else 'darkgreen')
                axes[col + 2].set_title(f"{title}\nMAE:{metrics['mae']:.1f}", fontsize=9, fontweight='bold', color=color)
            except:
                axes[col + 2].text(0.5, 0.5, 'Hata', ha='center', va='center')
        else:
            axes[col + 2].text(0.5, 0.5, 'Model Yok', ha='center', va='center')
        axes[col + 2].axis('off')
    
    plt.suptitle(f'🧪 TEST 3: {curve_type.upper()} #{img_idx+1}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/test3_{curve_type}_{img_idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

for model in loaded_models.values():
    if model: del model
torch.cuda.empty_cache()
print(f"✅ TEST 3 TAMAMLANDI")

---
## 🧪 TEST 4: FALLING (Düşen)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🧪 TEST 4: FALLING - MODEL TAHMİN KARŞILAŞTIRMASI
# ══════════════════════════════════════════════════════════════

print("="*70)
print("🧪 TEST 4: FALLING - MODEL TAHMİN KARŞILAŞTIRMASI")
print("="*70)

curve_type = 'falling'
n_images = IMAGES_PER_TEST
n_models = len(models_to_test)

print("\n📦 Modeller yükleniyor...")
loaded_models = {}
for m in models_to_test:
    try:
        loaded_models[m['name']] = load_model(m['path'], device)
    except:
        loaded_models[m['name']] = None

print(f"🎨 {n_images} sentetik görüntü test ediliyor...\n")

for img_idx in range(n_images):
    cfg = random_config(curve_type=curve_type)
    img, target, _ = draw_chart_matplotlib(cfg, W=IMG_SIZE, H=IMG_SIZE)
    img_noisy = add_scan_artifacts(img, strength=1.0)
    
    fig, axes = plt.subplots(1, n_models + 2, figsize=(4 * (n_models + 2), 5))
    
    axes[0].imshow(img_noisy)
    axes[0].set_title('📥 INPUT\n(Gürültülü Grafik)', fontsize=11, fontweight='bold', color='blue')
    axes[0].axis('off')
    
    axes[1].imshow(target)
    axes[1].set_title('🎯 GROUND TRUTH\n(Hedef Mask)', fontsize=11, fontweight='bold', color='green')
    axes[1].axis('off')
    
    for col, m in enumerate(models_to_test):
        model = loaded_models.get(m['name'])
        if model is not None:
            try:
                pred_raw, pred_vis = predict(model, img_noisy, device)
                metrics = calculate_metrics(pred_raw, target)
                axes[col + 2].imshow(pred_vis)
                title = f"🏆 PRED\n{m['source']} BEST" if 'BEST' in m['name'] else f"📤 PRED\n{m['name']}"
                color = 'purple' if 'BEST' in m['name'] else ('darkblue' if 'Kaggle' in m['name'] else 'darkgreen')
                axes[col + 2].set_title(f"{title}\nMAE:{metrics['mae']:.1f}", fontsize=9, fontweight='bold', color=color)
            except:
                axes[col + 2].text(0.5, 0.5, 'Hata', ha='center', va='center')
        else:
            axes[col + 2].text(0.5, 0.5, 'Model Yok', ha='center', va='center')
        axes[col + 2].axis('off')
    
    plt.suptitle(f'🧪 TEST 4: {curve_type.upper()} #{img_idx+1}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/test4_{curve_type}_{img_idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

for model in loaded_models.values():
    if model: del model
torch.cuda.empty_cache()
print(f"✅ TEST 4 TAMAMLANDI")

---
## 🧪 TEST 5: WAVY (Dalgalı)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🧪 TEST 5: WAVY - MODEL TAHMİN KARŞILAŞTIRMASI
# ══════════════════════════════════════════════════════════════

print("="*70)
print("🧪 TEST 5: WAVY - MODEL TAHMİN KARŞILAŞTIRMASI")
print("="*70)

curve_type = 'wavy'
n_images = IMAGES_PER_TEST
n_models = len(models_to_test)

print("\n📦 Modeller yükleniyor...")
loaded_models = {}
for m in models_to_test:
    try:
        loaded_models[m['name']] = load_model(m['path'], device)
    except:
        loaded_models[m['name']] = None

print(f"🎨 {n_images} sentetik görüntü test ediliyor...\n")

for img_idx in range(n_images):
    cfg = random_config(curve_type=curve_type)
    img, target, _ = draw_chart_matplotlib(cfg, W=IMG_SIZE, H=IMG_SIZE)
    img_noisy = add_scan_artifacts(img, strength=1.0)
    
    fig, axes = plt.subplots(1, n_models + 2, figsize=(4 * (n_models + 2), 5))
    
    axes[0].imshow(img_noisy)
    axes[0].set_title('📥 INPUT\n(Gürültülü Grafik)', fontsize=11, fontweight='bold', color='blue')
    axes[0].axis('off')
    
    axes[1].imshow(target)
    axes[1].set_title('🎯 GROUND TRUTH\n(Hedef Mask)', fontsize=11, fontweight='bold', color='green')
    axes[1].axis('off')
    
    for col, m in enumerate(models_to_test):
        model = loaded_models.get(m['name'])
        if model is not None:
            try:
                pred_raw, pred_vis = predict(model, img_noisy, device)
                metrics = calculate_metrics(pred_raw, target)
                axes[col + 2].imshow(pred_vis)
                title = f"🏆 PRED\n{m['source']} BEST" if 'BEST' in m['name'] else f"📤 PRED\n{m['name']}"
                color = 'purple' if 'BEST' in m['name'] else ('darkblue' if 'Kaggle' in m['name'] else 'darkgreen')
                axes[col + 2].set_title(f"{title}\nMAE:{metrics['mae']:.1f}", fontsize=9, fontweight='bold', color=color)
            except:
                axes[col + 2].text(0.5, 0.5, 'Hata', ha='center', va='center')
        else:
            axes[col + 2].text(0.5, 0.5, 'Model Yok', ha='center', va='center')
        axes[col + 2].axis('off')
    
    plt.suptitle(f'🧪 TEST 5: {curve_type.upper()} #{img_idx+1}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/test5_{curve_type}_{img_idx+1}.png', dpi=150, bbox_inches='tight')
    plt.show()

for model in loaded_models.values():
    if model: del model
torch.cuda.empty_cache()
print(f"✅ TEST 5 TAMAMLANDI")

---
## 📊 GENEL ÖZET

In [ ]:
# ══════════════════════════════════════════════════════════════
# 📊 SENTETİK TEST ÖZET RAPORU
# ══════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("📊 SENTETİK TEST ÖZET RAPORU")
print("="*70)

print(f"\n🔧 Test Parametreleri:")
print(f"   • Dataset: {DATASET_ROOT}")
print(f"   • Test edilen modeller: {len(models_to_test)}")
print(f"   • Her test için görüntü: {IMAGES_PER_TEST}")
print(f"   • Toplam test görüntüsü: {5 * IMAGES_PER_TEST}")

print(f"\n📁 Kaydedilen dosyalar:")
for f in glob.glob(f'{OUTPUT_DIR}/*.png'):
    print(f"   • {os.path.basename(f)}")

print("\n" + "="*70)
print("✅ SENTETİK TESTLER TAMAMLANDI!")
print("="*70)

---
## 🖼️ TEST 6: GERÇEK GRAFİKLER (Real Graphs)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🖼️ TEST 6: GERÇEK GRAFİKLER - MODEL KARŞILAŞTIRMA
# ══════════════════════════════════════════════════════════════

print("="*70)
print("🖼️ TEST 6: GERÇEK GRAFİKLER - MODEL TAHMİN KARŞILAŞTIRMASI")
print("="*70)

# Gerçek görüntüleri bul
real_images = []
if os.path.exists(REAL_GRAPHS_DIR):
    for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']:
        real_images.extend(glob.glob(f'{REAL_GRAPHS_DIR}/{ext}'))
        real_images.extend(glob.glob(f'{REAL_GRAPHS_DIR}/**/{ext}', recursive=True))
    real_images = list(set(real_images))
    real_images.sort()

if not real_images:
    print(f"⚠️ Gerçek görüntü bulunamadı: {REAL_GRAPHS_DIR}")
    print("   Klasöre .png, .jpg veya .jpeg dosyaları ekleyin.")
else:
    print(f"✅ {len(real_images)} gerçek görüntü bulundu")
    print(f"🔬 {len(models_to_test)} model test edilecek\n")
    
    # Tüm modelleri bir kere yükle (memory'de tut)
    loaded_models = {}
    print("📦 Modeller yükleniyor...")
    for m in models_to_test:
        try:
            loaded_models[m['name']] = load_model(m['path'], device)
            print(f"   ✅ {m['name']}")
        except Exception as e:
            print(f"   ❌ {m['name']}: {e}")
            loaded_models[m['name']] = None
    
    print(f"\n{'='*70}")
    
    # Her görüntü için test
    for img_idx, img_path in enumerate(real_images):
        img_filename = os.path.basename(img_path)
        print(f"\n📷 [{img_idx+1}/{len(real_images)}] {img_filename}")
        
        # Görüntüyü yükle
        try:
            img_pil = Image.open(img_path).convert('RGB')
            img_rgb = np.array(img_pil)
            img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        except Exception as e:
            print(f"   ❌ Yükleme hatası: {e}")
            continue
        
        n_models = len(models_to_test)
        
        # Figure: 1 satır - Input + tüm model tahminleri
        fig, axes = plt.subplots(1, n_models + 1, figsize=(4 * (n_models + 1), 5))
        
        # ═══ INPUT GÖRÜNTÜsü ═══
        axes[0].imshow(img_resized)
        axes[0].set_title(f'📥 INPUT\n(Orijinal Grafik)', fontsize=11, fontweight='bold', color='blue')
        axes[0].axis('off')
        # Çerçeve ekle
        for spine in axes[0].spines.values():
            spine.set_visible(True)
            spine.set_color('blue')
            spine.set_linewidth(3)
        
        # ═══ HER MODEL İÇİN TAHMİN ═══
        for col, m in enumerate(models_to_test):
            model = loaded_models.get(m['name'])
            
            if model is not None:
                try:
                    # TAHMİN YAP
                    pred_raw, pred_vis = predict(model, img_resized, device)
                    
                    # Göster
                    axes[col + 1].imshow(pred_vis)
                    
                    # Başlık: Model adı + kaynak
                    source_emoji = '🟦' if 'Kaggle' in m['name'] else '🟩'
                    if 'BEST' in m['name']:
                        title_color = 'green'
                        title = f"🏆 BEST MODEL\n({m['source']})"
                    else:
                        title_color = 'darkblue' if 'Kaggle' in m['name'] else 'darkgreen'
                        title = f"📤 PREDICTION\n{m['name']}"
                    
                    axes[col + 1].set_title(title, fontsize=10, fontweight='bold', color=title_color)
                    axes[col + 1].axis('off')
                    
                    print(f"   ✅ {m['name']} - Tahmin yapıldı")
                    
                except Exception as e:
                    axes[col + 1].text(0.5, 0.5, f'HATA:\n{str(e)[:30]}', 
                                       ha='center', va='center', fontsize=9,
                                       transform=axes[col + 1].transAxes, color='red')
                    axes[col + 1].set_title(f"❌ {m['name']}\n(Hata)", fontsize=10, color='red')
                    axes[col + 1].axis('off')
                    print(f"   ❌ {m['name']} - Hata: {e}")
            else:
                axes[col + 1].text(0.5, 0.5, 'Model\nYüklenemedi', 
                                   ha='center', va='center', fontsize=10, color='gray',
                                   transform=axes[col + 1].transAxes)
                axes[col + 1].set_title(f"⚠️ {m['name']}\n(Yüklenemedi)", fontsize=10, color='gray')
                axes[col + 1].axis('off')
        
        # Genel başlık
        plt.suptitle(f'🖼️ Real Graph Test: {img_filename}\n'
                     f'Input → Model Predictions (Soldan Sağa: Input, ardından {n_models} model tahmini)', 
                     fontsize=13, fontweight='bold', y=1.02)
        
        plt.tight_layout()
        
        # Kaydet
        save_name = os.path.splitext(img_filename)[0]
        save_path = f'{OUTPUT_DIR}/real_test_{save_name}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.show()
        
        print(f"   💾 Kaydedildi: {os.path.basename(save_path)}")
    
    # Memory temizle
    for name, model in loaded_models.items():
        if model is not None:
            del model
    torch.cuda.empty_cache()
    
    print("\n" + "="*70)
    print(f"✅ {len(real_images)} GERÇEK GRAFİK TEST EDİLDİ!")
    print(f"📁 Sonuçlar: {OUTPUT_DIR}")
    print("="*70)

---
## 📊 GENEL ÖZET

In [ ]:
# ══════════════════════════════════════════════════════════════
# 📊 GENEL ÖZET RAPORU
# ══════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("📊 GENEL TEST ÖZET RAPORU")
print("="*70)

print(f"\n🔧 Test Konfigürasyonu:")
print(f"   • Dataset: {DATASET_ROOT}")
print(f"   • Kaggle checkpoint'leri: {KAGGLE_INPUT_DIR}")
print(f"   • Colab checkpoint'leri: {COLAB_INPUT_DIR}")
print(f"   • Gerçek grafikler: {REAL_GRAPHS_DIR}")

print(f"\n📈 Test İstatistikleri:")
print(f"   • Test edilen modeller: {len(models_to_test)}")
print(f"   • Sentetik test türü: 5 (peaked_oval, peaked, rising, falling, wavy)")
print(f"   • Her sentetik test için görüntü: {IMAGES_PER_TEST}")
print(f"   • Toplam sentetik görüntü: {5 * IMAGES_PER_TEST}")
print(f"   • Gerçek grafik sayısı: {len(real_images) if 'real_images' in dir() else 'N/A'}")

print(f"\n📁 Kaydedilen tüm dosyalar:")
all_outputs = sorted(glob.glob(f'{OUTPUT_DIR}/*.png'))
for f in all_outputs:
    print(f"   • {os.path.basename(f)}")

print(f"\n📦 Çıktı klasörü: {OUTPUT_DIR}")
print(f"   Toplam dosya: {len(all_outputs)}")

print("\n" + "="*70)
print("🎉 TÜM TESTLER TAMAMLANDI!")
print("="*70)